In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Day 2 — AITR Transformer Fusion

Based on the MUSE paper ("Similarity over Factuality"). They show that replacing the MLP fusion with an **Attentive Intermediate Transformer Representations (AITR)** architecture pushes performance from 90% (MUSE-MLP) to 93.3% on NewsCLIPpings.

**Key insight:** transformer self-attention across feature representations beats simple concatenation + MLP, because it learns which feature dimensions to attend to *conditionally on the input*.

**What we feed the transformer:**
- Image CLIP embedding (768-d) → token 1
- Text CLIP embedding (768-d) → token 2
- Image-text element-wise product (768-d) → token 3 (RED-DOT-style fusion)
- Image-text element-wise difference (768-d) → token 4
- Scalar signals projected to 768-d → token 5 (CLIP probs/sims, evidence s2-s6, wiki, deberta)

Self-attention learns to weight these 5 tokens, then a final classification head predicts real/fake.

**No GPU strictly needed** — model is small (~2M params) but uses GPU if available.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = str(_cfg.ROOT)
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')
FUSION_SAVE   = os.path.join(PROJECT_ROOT, 'fusion_aitr')
os.makedirs(FUSION_SAVE, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


In [2]:
# ── Load CLIP embeddings (raw 768-dim) ──
print('Loading CLIP embeddings...')
img_feats = torch.load(os.path.join(CLIP_FEATURES, 'clip_img_features.pt'))
txt_feats = torch.load(os.path.join(CLIP_FEATURES, 'clip_txt_features.pt'))

# L2-normalize (they aren't normalized as saved)
img_feats = F.normalize(img_feats, dim=-1)
txt_feats = F.normalize(txt_feats, dim=-1)

print(f'Image: {img_feats.shape}  norms (after normalize): {img_feats.norm(dim=-1).mean():.4f}')
print(f'Text : {txt_feats.shape}  norms (after normalize): {txt_feats.norm(dim=-1).mean():.4f}')

# ── Load labels and scalar signals ──
id_df = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels = id_df['label'].values

clip_probs = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))

deb_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv'))
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

ev_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv'))
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup   = {row['id']: row for _, row in ev_df.iterrows()}
s2 = np.array([ev_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([ev_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([ev_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([ev_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([ev_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

wiki_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup   = {row['id']: row for _, row in wiki_df.iterrows()}
w1 = np.array([wiki_lookup.get(i, {}).get('wiki_score',     0.33) for i in id_df['id']])

# ── Stack all scalar signals ──
scalar_feats = np.column_stack([
    clip_probs, clip_sims, deb_scores, s2, s3, s4, s5, s6, w1
])  # (5000, 9)
scalar_feats = StandardScaler().fit_transform(scalar_feats)
scalar_feats = torch.tensor(scalar_feats, dtype=torch.float32)

print(f'Scalar features: {scalar_feats.shape}')
print(f'Labels: real={(labels==0).sum()} fake={(labels==1).sum()}')

Loading CLIP embeddings...
Image: torch.Size([5000, 768])  norms (after normalize): 1.0000
Text : torch.Size([5000, 768])  norms (after normalize): 1.0000
Scalar features: torch.Size([5000, 9])
Labels: real=2500 fake=2500


In [3]:
# ── AITR-style transformer model ──
# Architecture inspired by MUSE-AITR (Papadopoulos et al., 2024)

class AITR(nn.Module):
    def __init__(self, embed_dim=768, scalar_dim=9, num_heads=8, num_layers=2, 
                 dropout=0.3, hidden_dim=256):
        super().__init__()
        
        # Project scalar signals to embed_dim
        self.scalar_proj = nn.Sequential(
            nn.Linear(scalar_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )
        
        # Token type embeddings (5 token types: img, txt, prod, diff, scalar)
        self.type_embedding = nn.Embedding(5, embed_dim)
        
        # Transformer encoder for cross-token attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = embed_dim,
            nhead           = num_heads,
            dim_feedforward = embed_dim * 2,
            dropout         = dropout,
            activation      = 'gelu',
            batch_first     = True,
            norm_first      = True,  # pre-norm — more stable training
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # CLS-style aggregation: learnable query token
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, img_emb, txt_emb, scalar):
        B = img_emb.size(0)
        
        # Build 5 tokens per sample
        prod_emb   = img_emb * txt_emb              # (B, 768) interaction
        diff_emb   = img_emb - txt_emb              # (B, 768) mismatch
        scalar_emb = self.scalar_proj(scalar)       # (B, 768)
        
        tokens = torch.stack([
            img_emb, txt_emb, prod_emb, diff_emb, scalar_emb
        ], dim=1)  # (B, 5, 768)
        
        # Add type embeddings
        type_ids = torch.arange(5, device=tokens.device).unsqueeze(0).expand(B, -1)
        tokens   = tokens + self.type_embedding(type_ids)
        
        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)      # (B, 1, 768)
        tokens = torch.cat([cls, tokens], dim=1)    # (B, 6, 768)
        
        # Transformer
        out = self.transformer(tokens)              # (B, 6, 768)
        
        # Use CLS token output
        cls_out = out[:, 0]                          # (B, 768)
        
        return self.classifier(cls_out).squeeze(-1)

model = AITR(embed_dim=768, scalar_dim=9, num_heads=8, num_layers=2,
             dropout=0.3, hidden_dim=256).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: AITR transformer')
print(f'Parameters: {n_params:,} ({n_params/1e6:.2f}M)')

Model: AITR transformer
Parameters: 9,666,561 (9.67M)


In [4]:
# ── Train/val split + DataLoaders ──
indices = np.arange(len(labels))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels)

img_train, img_val = img_feats[train_idx], img_feats[val_idx]
txt_train, txt_val = txt_feats[train_idx], txt_feats[val_idx]
scl_train, scl_val = scalar_feats[train_idx], scalar_feats[val_idx]
y_train, y_val = labels[train_idx], labels[val_idx]

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)

train_ds = TensorDataset(img_train, txt_train, scl_train, y_train_t)
val_ds   = TensorDataset(img_val,   txt_val,   scl_val,   y_val_t)

train_ld = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
val_ld   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)

print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

Train: 4000 | Val: 1000


In [5]:
# ── Training loop ──
EPOCHS    = 100
PATIENCE  = 15
LR        = 5e-5     # transformers train slower than MLPs — small lr
WD        = 1e-4

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
criterion = nn.BCEWithLogitsLoss()

best_f1, best_epoch, patience_ctr, best_weights = 0.0, 0, 0, None

print(f'{"Epoch":>6} | {"Loss":>8} | {"Val F1":>8} | {"Val Acc":>8} | Status')
print('-' * 64)

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    n_batches = 0
    for img, txt, scl, yb in train_ld:
        img, txt, scl, yb = img.to(device), txt.to(device), scl.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(img, txt, scl)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_sum += loss.item()
        n_batches += 1
    scheduler.step()
    avg_loss = loss_sum / n_batches
    
    # Validation
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for img, txt, scl, yb in val_ld:
            img, txt, scl = img.to(device), txt.to(device), scl.to(device)
            logits = model(img, txt, scl)
            preds  = (torch.sigmoid(logits) > 0.5).long().cpu().numpy()
            preds_all.extend(preds)
            labels_all.extend(yb.long().numpy())
    
    vf1  = f1_score(labels_all, preds_all)
    vacc = accuracy_score(labels_all, preds_all)
    
    if vf1 > best_f1:
        best_f1, best_epoch, patience_ctr = vf1, epoch, 0
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        status = '** BEST **'
    else:
        patience_ctr += 1
        status = f'patience {patience_ctr}/{PATIENCE}'
    
    if epoch % 5 == 0 or 'BEST' in status:
        print(f'{epoch:>6} | {avg_loss:>8.4f} | {vf1:>8.4f} | {vacc:>8.4f} | {status}')
    
    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(best_weights)
print(f'\nBest Val F1: {best_f1:.4f} at epoch {best_epoch}')

 Epoch |     Loss |   Val F1 |  Val Acc | Status
----------------------------------------------------------------
     1 |   0.4145 |   0.8697 |   0.8690 | ** BEST **
     2 |   0.3366 |   0.8723 |   0.8680 | ** BEST **
     3 |   0.3151 |   0.8746 |   0.8670 | ** BEST **
     5 |   0.3048 |   0.8736 |   0.8680 | patience 2/15
     7 |   0.3010 |   0.8799 |   0.8750 | ** BEST **
    10 |   0.2927 |   0.8690 |   0.8580 | patience 3/15
    15 |   0.2818 |   0.8610 |   0.8540 | patience 8/15
    20 |   0.2801 |   0.8676 |   0.8620 | patience 13/15
Early stopping at epoch 22

Best Val F1: 0.8799 at epoch 7


In [6]:
# ── Final evaluation ──
model.eval()
all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for img, txt, scl, yb in val_ld:
        img, txt, scl = img.to(device), txt.to(device), scl.to(device)
        logits = model(img, txt, scl)
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(yb.long().numpy())

final_acc = accuracy_score(all_labels, all_preds)
final_f1  = f1_score(all_labels, all_preds)
final_auc = roc_auc_score(all_labels, all_probs)

print('=' * 65)
print('AITR — FINAL EVALUATION')
print('=' * 65)
print(f'Accuracy : {final_acc*100:.2f}%')
print(f'F1       : {final_f1:.4f}')
print(f'AUC-ROC  : {final_auc:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=['REAL', 'FAKE']))

print()
print('=' * 65)
print('PROGRESSION')
print('=' * 65)
results = [
    ('Your CLIP alone (v2)',                  85.60),
    ('+ DeBERTa + Evidence (MLP)',            87.40),
    ('+ Engineered features (XGBoost)',       86.20),
    ('+ AITR transformer (Day 2)',            final_acc * 100),
    ('--- Targets ---',                        0),
    ('SNIFFER',                               88.40),
    ('MUSE-MLP',                              90.00),
    ('RED-DOT',                               90.30),
    ('MUSE-AITR',                             93.30),
]
for name, val in results:
    if val == 0:
        print(f'  {name}')
    else:
        marker = ' <-- YOU' if 'Day 2' in name else ''
        print(f'  {name:<42} {val:.2f}%{marker}')

if final_acc * 100 > 88.4:
    print('\n*** Beats SNIFFER ***')
if final_acc * 100 > 90.0:
    print('*** Beats MUSE-MLP ***')

AITR — FINAL EVALUATION
Accuracy : 87.50%
F1       : 0.8799
AUC-ROC  : 0.9480

              precision    recall  f1-score   support

        REAL       0.91      0.83      0.87       500
        FAKE       0.85      0.92      0.88       500

    accuracy                           0.88      1000
   macro avg       0.88      0.88      0.87      1000
weighted avg       0.88      0.88      0.87      1000


PROGRESSION
  Your CLIP alone (v2)                       85.60%
  + DeBERTa + Evidence (MLP)                 87.40%
  + Engineered features (XGBoost)            86.20%
  + AITR transformer (Day 2)                 87.50% <-- YOU
  --- Targets ---
  SNIFFER                                    88.40%
  MUSE-MLP                                   90.00%
  RED-DOT                                    90.30%
  MUSE-AITR                                  93.30%


In [7]:
# ── Save model ──
import json
torch.save(model.state_dict(), os.path.join(FUSION_SAVE, 'aitr_weights.pt'))
summary = {
    'final_acc': float(final_acc),
    'final_f1' : float(final_f1),
    'final_auc': float(final_auc),
    'best_epoch': int(best_epoch),
    'architecture': 'AITR transformer',
    'embed_dim': 768,
    'scalar_dim': 9,
    'tokens': ['img', 'txt', 'img*txt', 'img-txt', 'scalar'],
}
with open(os.path.join(FUSION_SAVE, 'results.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved.')
print(json.dumps(summary, indent=2))

Saved.
{
  "final_acc": 0.875,
  "final_f1": 0.8799231508165226,
  "final_auc": 0.9479599999999999,
  "best_epoch": 7,
  "architecture": "AITR transformer",
  "embed_dim": 768,
  "scalar_dim": 9,
  "tokens": [
    "img",
    "txt",
    "img*txt",
    "img-txt",
    "scalar"
  ]
}
